# The same process, a country where names stop working

**Nigeria schools: two independent surveys of the same schools, reconciled the
way notebook 13 reconciled England's.**

[Notebook 13](13_england_schools.ipynb) crosswalked the DfE register against
OpenStreetMap in Leeds and found something uncomfortable: on F1, **plain exact
name matching was level with the shipped pack**. English school names are
unusually standardised, so a string comparison is nearly enough.

This runs the same process on Nigerian schools and the result inverts. Exact
matching goes from the safest method available to the most dangerous one.

Nothing about the engine changes between the two notebooks. What changes is
what a name is worth.

## The two sources

The GRID3 schools register for Nigeria is an aggregate: six surveys merged into
one file, each having visited schools independently.

| source | records |
|---|---|
| NMIS | 78,770 |
| GRID | 17,345 |
| OSGOF | 8,166 |
| eHA Polio | 2,389 |

**NMIS** and **GRID** both cover 24 states, and neither carries the other's
identifiers. That is the same shape as GIAS against OpenStreetMap: two parties
who surveyed the same world and cannot join their results.

One state keeps this readable. Enugu has 1,776 NMIS records and 1,504 GRID
records, which is 2.7 million candidate pairs before blocking.

In [1]:
import csv, math, random, re, statistics, sys
from collections import Counter, defaultdict
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "packages" / "arche-core").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "packages" / "arche-core" / "src"))

CSV = REPO / "data" / "_cache" / "schools" / "nigeria_schools.csv"
if not CSV.exists():
    raise SystemExit(
        "Stage the register first (fetched, never committed):\n"
        "    python data/scripts/stage_nigeria_schools.py --csv <download>.csv"
    )

rows = [r for r in csv.DictReader(CSV.open(encoding="utf-8-sig"))
        if (r.get("name") or "").strip()]
STATE = "Enugu"
nmis = [r for r in rows if r["source"].strip() == "NMIS" and r["statename"].strip() == STATE]
grid = [r for r in rows if r["source"].strip() == "GRID" and r["statename"].strip() == STATE]
print(f"register    : {len(rows):,} named records")
print(f"{STATE} NMIS  : {len(nmis):>6,}")
print(f"{STATE} GRID  : {len(grid):>6,}")
print(f"candidates  : {len(nmis)*len(grid):,} before blocking")

register    : 107,670 named records
Enugu NMIS  :  1,776
Enugu GRID  :  1,504
candidates  : 2,671,104 before blocking


## Step 1 — There is no truth set, and saying so is the honest start

Notebook 13 had 282 labels, because OpenStreetMap editors had written
`ref:edubase` tags asserting *this mapped school is that URN*. Nothing
equivalent exists here. NMIS and GRID never linked their records.

So this notebook **cannot report recall**, and any table that did would be
inventing labels.

What it can report is the error that matters, because one label is free and
certain:

> **Two schools in different states are not the same school.**

That gives negatives nobody constructed. Every method below is scored on pairs
it should never merge.

## Step 2 — Why the names are the story

Before matching anything, look at what a Nigerian school name contains.

In [2]:
names = Counter(r["name"].strip().upper() for r in rows)
shared = {n: c for n, c in names.items() if c > 1}
print(f"distinct names         {len(names):>8,} of {len(rows):,}")
print(f"names held by >1 school{len(shared):>8,}")
print(f"records sharing a name {sum(shared.values()):>8,}  ({100*sum(shared.values())/len(rows):.0f}%)")
print()
for n, c in names.most_common(6):
    st = len({r["statename"] for r in rows if r["name"].strip().upper() == n})
    print(f"  {c:>4}x  {n[:46]:<46} across {st} states")

distinct names           98,248 of 107,670
names held by >1 school   3,959
records sharing a name   13,381  (12%)

   200x  COMMUNITY PRIMARY SCHOOL                       across 21 states
   120x  LGEA PRIMARY SCHOOL                            across 11 states
   107x  GOVERNMENT JUNIOR SECONDARY SCHOOL             across 15 states


    99x  NOMADIC PRIMARY SCHOOL                         across 24 states
    92x  COMMUNITY SECONDARY SCHOOL                     across 22 states


    86x  LOCAL EDUCATION AUTHORITY PRIMARY SCHOOL       across 10 states


Two hundred schools are called `COMMUNITY PRIMARY SCHOOL`, spread over 21
states. Ninety-nine are called `NOMADIC PRIMARY SCHOOL`, over 24.

Compare that with Leeds, where a school called `Roundhay School` is the only
one. **One in eight Nigerian records shares its name with a different school.**

The name is built from words describing what the school *is*, not which one it
is. `COMMUNITY`, `PRIMARY`, `SCHOOL`, `LGEA`, `NOMADIC` are all category words.
A string comparator reading two of these sees near-perfect agreement and is
not wrong about the strings. It is wrong about what agreement is worth.

## Step 3 — The certain negatives

In [3]:
by_name = defaultdict(list)
for r in rows:
    by_name[r["name"].strip().upper()].append(r)

rng = random.Random(20260819)
pool = sorted(n for n, rs in by_name.items() if len({x["statename"] for x in rs}) > 1)
rng.shuffle(pool)

negatives = []
for n in pool:
    if len(negatives) >= 400:
        break
    rs = sorted(by_name[n], key=lambda x: x["uniq_id"])
    a = rs[0]
    b = next((x for x in rs if x["statename"] != a["statename"]), None)
    if b:
        negatives.append((a, b))

print(f"{len(negatives)} pairs that share a name exactly, in different states.")
print()
for a, b in negatives[:4]:
    print(f"  {a['name'][:40]:<40} {a['statename']:<12} vs {b['statename']}")

400 pairs that share a name exactly, in different states.



  Golden Nursery and Primary School        Delta        vs Akwa Ibom
  Zakkam Primary School                    Bauchi       vs Plateau
  Dorawa Primary School                    Bauchi       vs Sokoto
  Gada Biyu Primary School                 Bauchi       vs Kano


## Step 4 — The same four methods as notebook 13

In [4]:
from rapidfuzz import fuzz
from arche.resolve import crosswalk

def toks(s): return {t for t in re.split(r"[^a-z0-9]+", s.casefold()) if t}
def jaccard(x, y):
    tx, ty = toks(x), toks(y)
    return len(tx & ty) / len(tx | ty) if tx and ty else 0.0

def rec(r):
    out = {"id": r["uniq_id"], "name": r["name"].strip()}
    try:
        out["lat"], out["lon"] = str(float(r["y"])), str(float(r["x"]))
    except (TypeError, ValueError):
        pass
    return out

rowsout = []
for label, fn in (
    ("exact name (casefold)", lambda a, b: a.casefold().strip() == b.casefold().strip()),
    ("token Jaccard >= 0.5",  lambda a, b: jaccard(a, b) >= 0.5),
    ("token_set_ratio >= 90", lambda a, b: fuzz.token_set_ratio(a, b) >= 90),
):
    rowsout.append((label, sum(1 for a, b in negatives if fn(a["name"], b["name"]))))

pairs = {(a["uniq_id"], b["uniq_id"]) for a, b in negatives}
res = crosswalk([rec(a) for a, _ in negatives], [rec(b) for _, b in negatives],
                entity="place", id_field="id")
merged = sum(1 for e in res["matches"] if e["decision"] == "match" and (e["a_id"],
    e["b_id"]) in pairs)
held   = sum(1 for e in res["matches"] if e["decision"] == "review" and (e["a_id"],
    e["b_id"]) in pairs)
rowsout.append(("arche (name + coords)", merged))

print(f"{'method':<26}{'false merges':>14}{'rate':>9}")
for label, n in rowsout:
    print(f"{label:<26}{n:>14,}{n/len(negatives):>9.1%}")
print()
print(f"arche routed {held} of {len(negatives)} to review rather than deciding.")

method                      false merges     rate
exact name (casefold)                400   100.0%
token Jaccard >= 0.5                 400   100.0%
token_set_ratio >= 90                399    99.8%
arche (name + coords)                  2     0.5%

arche routed 397 of 400 to review rather than deciding.


### Read the construction honestly

These pairs were chosen *because* they share a name, so exact matching failing
all 400 is true by construction. That is not a trick, it is the finding, and it
only matters because of the number in step 2: **12% of records are exposed to
it.** In Leeds, two schools sharing a name exactly were nearly always the same
school. Here they are nearly always different ones.

Put the two notebooks side by side.

| | Leeds | Nigeria |
|---|---|---|
| exact name | precision 0.992, **2** false merges | **400 / 400** wrong |
| token_set_ratio >= 90 | 131 false merges | 399 / 400 wrong |
| arche, name + coords | 37 false merges | **2 / 400** wrong |

Same engine, same comparators, same four methods. The only thing that changed
is the naming culture.

## Step 5 — The actual reconciliation

The negatives measure safety. This is the job someone actually wants done:
link the two surveys.

In [5]:
A = [rec(r) for r in nmis]
B = [rec(r) for r in grid]
out = crosswalk(A, B, entity="place", id_field="id")
d = Counter(e["decision"] for e in out["matches"])
print(f"candidate pairs after blocking : {len(out['matches']):,}")
print(f"  match  {d.get('match', 0):>6,}")
print(f"  review {d.get('review', 0):>6,}")
print()
print("pins.tf =", out["pins"]["tf"])
print()
for e in [x for x in out["matches"] if x["decision"] == "match"][:6]:
    a = next(r for r in nmis if r["uniq_id"] == e["a_id"])
    b = next(r for r in grid if r["uniq_id"] == e["b_id"])
    print(f"  {e['score']:.3f}  {a['name'][:34]:<34} <-> {b['name'][:34]}")

candidate pairs after blocking : 3,639
  match     517
  review  3,122

pins.tf = shipped:place@sha256:c94f20a1c2dfba18+phrases@sha256:ed19b623d77407d3

  0.999  UPE School Ezza Mpu                <-> Upe School Ezza Mpu
  0.999  Trinity Nursery and Primary School <-> Trinity Nursery And Primary School
  0.998  Community Primary School Agu Ekweg <-> Community Primary School Ekwegbe A
  0.998  Community School Adogba Awgu       <-> Adogba Community School Awgu
  0.998  Community Secondary School Ezebuna <-> Ezebunagu Community Secondary Scho
  0.996  St Francis Nursery and Primary Sch <-> St Francis Nursery And Primary Sch


Without labels these are proposals, not verified links. That is what the review
queue is for, and why the engine puts so many of them there rather than
asserting them.

## What this measured, and what it did not

**Measured.** The false-merge behaviour of four methods on 400 pairs that are
certainly different schools, drawn from the register rather than written by us.

**Not measured.**

* **Recall.** No positive labels exist here. A method that refuses everything
  would score perfectly on this page. Notebook 13 is the control: the same
  engine on labelled English data reaches recall 0.986.
* **Whether the step 5 links are correct.** They are proposals. Nobody has
  adjudicated them.
* **arche is not clean either.** It merged 2 of the 400. Those two are worth
  reading, because they are the shape of what still gets through.

The pair of notebooks is the argument. Neither is complete on its own: 13
without 14 says representation buys you little, and 14 without 13 says a
matcher that abstains is safe. Both are true, and neither is the point.